In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
device = torch.device("cuda")
device

device(type='cuda')

In [5]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

GPT2Tokenizer(name_or_path='microsoft/DialoGPT-small', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [6]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small").to(device)
model

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

,context,response
0,i'm going through some things with my feelings...,if everyone thinks you're worthless then maybe...
1,i'm going through some things with my feelings...,hello and thank you for your question and seek...
2,i'm going through some things with my feelings...,first thing i'd suggest is getting the sleep y...
3,i'm going through some things with my feelings...,therapy is essential for those that are feelin...
4,i'm going through some things with my feelings...,i first want to let you know that you are not ...


In [8]:
contexts = dataset['context'].astype('str').values
contexts[:5]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthle

In [9]:
responses = dataset['response'].astype('str').values
responses[:5]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today",
       "hello and thank you for you

In [10]:
def combineText(example):
    return {
        "text": "User: " + example['context'] +
            "Bot: " + example['response']
    }

In [11]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=64)

In [12]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [13]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

Dataset({
    features: ['context', 'response'],
    num_rows: 3512
})

In [14]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['context', 'response'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['context', 'response'],
        num_rows: 703
    })
})

In [15]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['context', 'response'],
    num_rows: 2809
})

In [16]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['context', 'response'],
    num_rows: 703
})

In [17]:
trainSet = trainSet.map(combineText)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 2809
})

In [18]:
testSet = testSet.map(combineText)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 703
})

In [19]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 2809
})

In [20]:
testSet = testSet.map(encode, batched=True)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 703
})

In [21]:
trainSet = trainSet.map(add_labels)
trainSet

Map:   0%|          | 0/2809 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [22]:
testSet = testSet.map(add_labels)
testSet

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [23]:
print(set(trainSet[0]["labels"]))

{257, 3589, 3206, 779, 1037, 5141, 20630, 13721, 25, 284, 287, 1312, 290, 428, 2222, 3505, 307, 12982, 8119, 1595, 24636, 326, 19271, 8776, 12361, 2891, 460, 1613, 1101, 845, 5328, 2000, 2130, 3285, 1243, 31707, 25822, 351, 736, 355, 2279, 616, 617, 750, 2158, 1394, 502, 7926, 9846}


In [24]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy='epoch',
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    max_grad_norm = 1.0
)

In [25]:
# trainSet = trainSet.select(range(1100))

In [26]:
# testSet = testSet.select(range(670))

In [27]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [28]:
trainer.evaluate(testSet)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'eval_loss': 6.302911281585693,
 'eval_model_preparation_time': 0.0019,
 'eval_runtime': 5.8213,
 'eval_samples_per_second': 120.764,
 'eval_steps_per_second': 60.468}

In [29]:

trainer.predict(testSet)

PredictionOutput(predictions=array([[[ 25.19,  20.33,  19.92, ...,  23.66,  22.03,  24.33],
        [398.8 , 360.5 , 359.  , ..., 395.  , 398.8 , 410.  ],
        [370.5 , 332.5 , 329.5 , ..., 366.8 , 373.2 , 381.2 ],
        ...,
        [408.8 , 366.2 , 362.8 , ..., 403.2 , 408.2 , 425.2 ],
        [376.2 , 339.5 , 335.  , ..., 370.2 , 377.8 , 389.  ],
        [406.2 , 362.5 , 359.  , ..., 399.8 , 405.  , 422.5 ]],

       [[ 25.19,  20.33,  19.92, ...,  23.66,  22.03,  24.33],
        [398.8 , 360.5 , 359.  , ..., 395.  , 398.8 , 410.  ],
        [362.2 , 326.8 , 320.  , ..., 358.  , 361.8 , 370.2 ],
        ...,
        [417.  , 374.2 , 373.8 , ..., 415.2 , 417.  , 433.8 ],
        [396.2 , 354.8 , 354.5 , ..., 393.  , 395.5 , 411.2 ],
        [423.8 , 381.5 , 378.5 , ..., 421.2 , 423.5 , 442.2 ]],

       [[ 25.19,  20.33,  19.92, ...,  23.66,  22.03,  24.33],
        [398.8 , 360.5 , 359.  , ..., 395.  , 398.8 , 410.  ],
        [362.2 , 326.8 , 320.  , ..., 358.  , 361.8 , 370.2

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,0.000000,nan,0.001900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1405, training_loss=46.27795540480427, metrics={'train_runtime': 245.6427, 'train_samples_per_second': 11.435, 'train_steps_per_second': 5.72, 'total_flos': 133379455942656.0, 'train_loss': 46.27795540480427, 'epoch': 1.0})

In [31]:
trainer.save_model("mental-health-dialogpt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [32]:
trainer.evaluate(testSet)

{'eval_loss': nan,
 'eval_model_preparation_time': 0.0019,
 'eval_runtime': 3.7822,
 'eval_samples_per_second': 185.871,
 'eval_steps_per_second': 93.068,
 'epoch': 1.0}

In [33]:
trainer.predict(testSet)

PredictionOutput(predictions=array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       ...,

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan,